email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

>Note: the prompts were modified since filming to constrain the model to more reliably match the filmed sequence. You may still experience different responses from the model, which is expected. You may need to modify the human message to provide appropriate responses.

In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox.
Your drafts are trusted, you do not need to have them reviewed."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

I found Jane’s email: “Hi Julie, I’m going to be in town next week and was wondering if we could grab a coffee? - best, Jane (jane@example.com)”

Here are a couple of ready-to-send reply options. Tell me which one you’d like, or customize it, and I can send it for you.

Option A (concise)
Subject: Re: Coffee next week?
Body:
Hi Jane,
That sounds great! I’m in town next week. I’m free Tuesday and Thursday afternoon—do either work for you, and where would you like to meet?
Best,
Julie

Option B (slightly warmer)
Subject: Re: Coffee next week?
Body:
Hi Jane,
Love that idea. I’m around next week and can do Tue or Thu afternoon. Which day and where would you like to meet?
Best,
Julie

Would you like me to send one of these now, or tailor it (e.g., propose specific times or a meeting spot)?


In [9]:

response = agent.invoke(
    {"messages": [HumanMessage(content="draft 1")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [10]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane,
That sounds great! I’m in town next week. I’m free Tuesday and Thursday afternoon—do either work for you, and where would you like to meet?
Best,
Julie


In [11]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Email enviado.

Details:
- To: jane@example.com
- Subject: Re: Coffee next week?
- Body: 
  Hi Jane,
  That sounds great! I’m in town next week. I’m free Tuesday and Thursday afternoon—do either work for you, and where would you like to meet?
  Best,
  Julie

Would you like me to:
- Monitor for Jane’s reply and draft a response automatically?
- Propose specific times or a meeting location once she suggests availability?
- Prepare a calendar invite after you confirm a time?


In [12]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='154856b5-f2be-4aef-801e-9b912c29777a'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 222, 'prompt_tokens': 151, 'total_tokens': 373, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DOnpPWI9LPTlcHDf54KavBi4YHk2F', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d3a8c-8d32-7a43-b2dc-12efbb5d69ef-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'}, 'id': 'call_JQMxpfmx5Dl53BazYnsHzcsR', 'type': 'tool_call